# Claim Verification Pipeline

**Pipeline steps :**
1. Load data
2. Match our questions -> closest QuantTemp question (SBERT cosine similarity)
3. Extract target entities from matched QT question (GLiNER)
4. Fetch article; if inaccessible, walk fallback snippets
5. Build per-article FAISS vector DB
6. **Multi-query retrieval** -- snippet splits (by `...`) + individual target entities as queries
7. **Threshold logic** -- >0.7 swap, 0.5-0.7 keep both, <0.5 snippet only
8. **Entity extraction from chunks** (score > 0.5) + set comparison for TRUE/CONFLICTING signal
9. **Checkpoint saving** -- auto-saves after every N records; resume safely
10. **Empirical analysis** -- inspect 10-20 cases >= 0.6 sim: snippet vs chunk
11. Export enriched JSON

---
**Decision Logic :**
```
For each question:
  -> Multi-query FAISS (snippet splits + individual entities)
    |- Best chunk score > 0.7  -> swap snippet with chunk as evidence
    |- Best chunk score 0.5-0.7 -> keep chunk + keep original snippet
    '- Best chunk score < 0.5  -> fall back to snippet only

  -> Collect entities from all chunks with score > 0.5
    -> Compare against target entities
      |- Entities match + good chunk -> likely TRUE
      |- Good chunk + missing entities -> CONFLICTING / possibly FALSE
      '- No good chunk + missing entities -> CONFLICTING
```


## 0. Install & Imports

In [ ]:
!pip install -q gdown gliner trafilatura sentence-transformers faiss-cpu requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 32.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ast, json, os, re, time, warnings
from collections import defaultdict
from typing import List, Optional, Dict, Any

import numpy as np
import faiss
import requests
import torch
import trafilatura
from gliner import GLiNER
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# Config
# DATA_PATH        = "/content/Processed_complete_dataset.json"
DATA_PATH          = "/content/temporal_dataset_with_new_evidences.json"
MATCH_THRESHOLD  = 0.15
TOP_K_SNIPPETS   = 3
CHUNK_SIZE       = 3
CHUNK_OVERLAP    = 1
SBERT_MODEL      = "all-MiniLM-L6-v2"
GLINER_MODEL     = "urchade/gliner_small-v2.1"
GLINER_THRESHOLD = 0.30
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

# NEW thresholds from meeting
SCORE_HIGH = 0.95  # > this -> swap snippet with chunk
SCORE_MID  = 0.6  # > this -> keep chunk + snippet; < this -> snippet only

# Checkpoint paths
CHECKPOINT_PATH  = "/content/drive/MyDrive/claim_verification_analysis_updated_v2/checkpoint_records.json"
FINAL_OUTPUT_PATH = "/content/drive/MyDrive/claim_verification_analysis_updated_v2/claim_verification_results_v2.json"

print("Config ready. Device:", DEVICE)


## 1. Load Data

In [ ]:
!gdown --id 1jU264txlUGx4n0lWhb4VP41qBbPk-u_N

with open(DATA_PATH) as f:
    raw = json.load(f)

claims, labels, categories, taxonomy = {}, {}, {}, {}
quantemp_evidences, our_evidences, reranked_evidences = {}, {}, {}

for idx, item in enumerate(raw):
    sid = str(idx)
    claims[sid]             = item["claim"]
    labels[sid]             = item["label"]
    categories[sid]         = item["Category"]
    taxonomy[sid]           = item["taxonomy"]
    quantemp_evidences[sid] = item["quantemp_evidences"]
    our_evidences[sid]      = item["our_evidences"]
    reranked_evidences[sid] = item["reranked_our_evidences"]
sample_ids = list(claims.keys())
print(f"Loaded {len(sample_ids)} claims.")


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1jU264txlUGx4n0lWhb4VP41qBbPk-u_N
From (redirected): https://drive.google.com/uc?id=1jU264txlUGx4n0lWhb4VP41qBbPk-u_N&confirm=t&uuid=e634a918-5306-46fe-879a-3ee175059df8
To: /content/Processed_complete_dataset.json
100% 252M/252M [00:04<00:00, 54.8MB/s]
Loaded 4130 claims.


## 2. Load Models (SBERT + GLiNER)

In [ ]:
print(f"Loading SBERT ({SBERT_MODEL})...")
encoder = SentenceTransformer(SBERT_MODEL)

print(f"Loading GLiNER ({GLINER_MODEL}) on {DEVICE}...")
gliner = GLiNER.from_pretrained(GLINER_MODEL)
if DEVICE == "cuda":
    gliner = gliner.to(DEVICE)

print("Models ready")


## 3. Question Matching (Our -> QuantTemp)

In [ ]:
def encode(texts):
    return encoder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

def match_questions(our_qs, qt_qs, threshold=MATCH_THRESHOLD):
    if not qt_qs or not our_qs:
        return [{"our_question": q, "matched_qt_question": None, "score": 0.0} for q in our_qs]
    sim_mat = encode(our_qs) @ encode(qt_qs).T
    results = []
    for i, our_q in enumerate(our_qs):
        best_j   = int(np.argmax(sim_mat[i]))
        best_sim = float(sim_mat[i][best_j])
        results.append({
            "our_question":        our_q,
            "matched_qt_question": qt_qs[best_j] if best_sim >= threshold else None,
            "score":               round(best_sim, 4),
        })
    return results

match_results = {}
for sid in sample_ids:
    our_qs = [item["question"] for item in our_evidences[sid]]
    qt_qs  = [item["questions"].strip() for item in quantemp_evidences[sid]]
    match_results[sid] = match_questions(our_qs, qt_qs)

print("Question matching done.")


Question matching done.


## 4. Entity Extraction (GLiNER)

In [ ]:
ENTITY_TYPE_RULES = [
    (r"^who\b",                                                          "PERSON"),
    (r"^which (country|nation|state|government)",                        "GPE"),
    (r"^which (organization|agency|office|body|entity|group|party|committee)", "ORG"),
    (r"^which (program|scheme|policy|act|bill|law|legislation|plan)",    "POLICY_PRODUCT"),
    (r"^which (vaccine|drug|treatment|medication|medicine)",             "MEDICAL_PRODUCT"),
    (r"^which (company|brand|firm|corporation|startup)",                 "ORG"),
    (r"^which\b",                                                        "ENTITY_GENERIC"),
    (r"^what (year|date|time|month|day)",                                "DATE"),
    (r"^when\b",                                                         "DATE"),
    (r"^where\b",                                                        "LOCATION"),
    (r"^how (much|many|often|long|far|high|low)",                        "QUANTITY"),
    (r"^what\b",                                                         "ENTITY_GENERIC"),
]

ENTITY_TYPE_TO_GLINER_LABELS = {
    "PERSON":          ["person", "organization", "government agency", "company", "political party"],
    "GPE":             ["country", "government", "nation", "state", "political body", "city"],
    "ORG":             ["organization", "company", "political party", "government agency", "institution"],
    "POLICY_PRODUCT":  ["policy", "law", "act", "program", "legislation", "scheme", "bill"],
    "MEDICAL_PRODUCT": ["vaccine", "drug", "medication", "medical treatment", "medicine"],
    "DATE":            ["date", "year", "time period", "month"],
    "LOCATION":        ["location", "place", "country", "region", "city"],
    "QUANTITY":        ["percentage", "number", "quantity", "amount", "statistic"],
    "ENTITY_GENERIC":  ["organization", "person", "country", "product", "event", "entity"],
    "UNKNOWN":         ["person", "organization", "country", "location", "event", "product"],
}

_WIDE_LABELS = [
    "person", "organization", "country", "government", "location",
    "date", "number", "percentage", "vaccine", "drug", "law", "policy",
    "product", "event", "political party", "agency",
]

_STOPWORDS = {
    "the","a","an","is","are","was","were","has","have","had","did","do","does",
    "been","be","will","would","could","should","there","it","this","that","of",
    "in","to","for","on","at","by","with","about","or","and","if","than","any",
    "some","all","its","from","more","once","they","get","us","only","certain",
}

def infer_entity_type(question: str) -> str:
    q = question.strip().lower()
    for pattern, etype in ENTITY_TYPE_RULES:
        if re.search(pattern, q):
            return etype
    return "UNKNOWN"

def _gliner_predict(text, labels, threshold):
    try:
        return gliner.predict_entities(text, labels, threshold=threshold)
    except Exception:
        return []

def _dedupe_spans(spans, priority_map, n_labels):
    seen, entities = set(), []
    for span in spans:
        text = span["text"].strip()
        norm = text.lower()
        if norm in seen or len(text) <= 1 or norm in _STOPWORDS:
            continue
        seen.add(norm)
        entities.append({
            "text":     text,
            "label":    span["label"],
            "score":    round(float(span["score"]), 3),
            "priority": priority_map.get(span["label"], n_labels),
        })
    entities.sort(key=lambda e: (e["priority"], -e["score"]))
    return entities

def _noun_phrase_fallback(qt_question: str) -> str:
    q = re.sub(r"^(did|is|are|was|were|has|have|does|do|will|would|could|should)\s+", "", qt_question.strip().rstrip("?"), flags=re.I)
    q = re.sub(r"^(the|a|an)\s+", "", q, flags=re.I)
    phrase = [tok.strip(",.?!;:\"'()") for tok in q.split()
              if tok.strip(",.?!;:\"'()").lower() not in _STOPWORDS and len(tok) > 1][:6]
    return " ".join(phrase) if phrase else qt_question[:60]

def extract_target_entity(our_question: str, qt_question: str):
    if not qt_question or not qt_question.strip():
        return {"entity_type": "UNKNOWN", "best_entity": "", "entities": [], "status": "NO_QT"}
    etype   = infer_entity_type(our_question)
    labels  = ENTITY_TYPE_TO_GLINER_LABELS.get(etype, ENTITY_TYPE_TO_GLINER_LABELS["UNKNOWN"])
    pri_map = {lbl: i for i, lbl in enumerate(labels)}
    spans   = _gliner_predict(qt_question, labels, GLINER_THRESHOLD)
    ents    = _dedupe_spans(spans, pri_map, len(labels))
    if ents:
        status = "OK" if ents[0]["priority"] == 0 else "FALLBACK_LABEL"
        best   = ents[0]["text"]
    else:
        best   = _noun_phrase_fallback(qt_question)
        ents   = [{"text": best, "label": "noun_phrase", "score": 0.0, "priority": 999}]
        status = "FALLBACK_NP"
    return {"entity_type": etype, "best_entity": best, "entities": ents, "status": status}

def scan_all_entities(text: str) -> list:
    if not text or not text.strip():
        return []
    spans = _gliner_predict(text, _WIDE_LABELS, threshold=0.25)
    seen, results = set(), []
    for span in spans:
        norm = span["text"].strip().lower()
        if norm and norm not in seen and norm not in _STOPWORDS and len(norm) > 1:
            seen.add(norm)
            results.append((span["text"].strip(), span["label"]))
    return results

def compute_missing_entities(qt_entities, our_entities):
    our_norms = {t.lower().strip(",.?\"'") for t, _ in our_entities}
    return [
        (qt_text, qt_label)
        for qt_text, qt_label in qt_entities
        if not any(
            qt_text.lower().strip(",.?\"'") == o or
            qt_text.lower() in o or o in qt_text.lower()
            for o in our_norms if len(o) > 2
        )
    ]

def any_entity_in_text(entity_texts, text):
    text_l = text.lower()
    for ent in entity_texts:
        if not ent: continue
        if ent.lower() in text_l:
            return True, ent
        parts = [p for p in ent.lower().split() if p not in _STOPWORDS]
        if len(parts) >= 2 and sum(1 for p in parts if p in text_l) >= max(2, round(len(parts)*0.6)):
            return True, ent
    return False, None

print("Entity extraction helpers defined.")


Entity extraction helpers defined.


## 5. Article Fetching with Fallback

In [ ]:
import logging
import concurrent.futures

logging.getLogger("trafilatura").setLevel(logging.ERROR)

def fetch_article(url: str, timeout: int = 5) -> Optional[str]:
    if url.lower().endswith(".pdf") or "pdf" in url.lower():
        return None
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code != 200:
            return None
        ct = resp.headers.get("Content-Type", "")
        if "pdf" in ct.lower():
            return None
        return trafilatura.extract(resp.text) or None
    except Exception:
        return None

def fetch_best_article(question: str, top_link: str, fallback_results: list):
    text = fetch_article(top_link)
    if text:
        return text, top_link, "top"
    for i, candidate in enumerate(fallback_results):
        url = candidate.get("link", "")
        if not url or url == top_link:
            continue
        if url.lower().endswith(".pdf") or "pdf" in url.lower():
            continue
        text = fetch_article(url)
        if text:
            return text, url, f"fallback_rank_{i+1}"
        if i >= 2:
            break
    return None, None, "failed"

print("Article fetching helpers defined.")


Article fetching helpers defined.


## 6. FAISS Vector DB Helpers

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    sentences = [s.strip() for s in text.replace("\n", " ").split(". ") if len(s.strip()) > 20]
    step   = chunk_size - overlap
    chunks = []
    for start in range(0, len(sentences), step):
        window = sentences[start: start + chunk_size]
        if window:
            chunks.append(". ".join(window) + ".")
    return chunks

def build_faiss_index(chunks: list):
    if not chunks:
        return None, []
    embs  = encoder.encode(chunks, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, chunks

def query_index(index, chunks: list, query: str, top_k: int = 3) -> list:
    if index is None or not chunks:
        return []
    q_vec = encoder.encode([query], normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    scores, idxs = index.search(q_vec, min(top_k, len(chunks)))
    return [{"rank": r+1, "score": round(float(scores[0][r]), 4), "text": chunks[idxs[0][r]]}
            for r in range(len(idxs[0]))]

print("FAISS helpers defined.")


FAISS helpers defined.


## 7. Multi-Query Retrieval Helpers

**From meeting:**
- Using only the question as query gives small semantic scores -> poor retrieval.
- Instead, query FAISS in multiple ways:
  1. **Snippet splits**: Google snippets use `...` (triple dots). Split by `...` and use each part as a query.
  2. **Individual target entities**: e.g., "Congressional", "Obamacare", "US" each become separate queries.
- Collect chunks from all queries and merge them.


In [ ]:
def split_snippet_queries(snippet: str) -> list:
    """
    Split a Google-style snippet by triple dots (...) to get focused sub-queries.
    Google provides the text where the query matches in the page, separated by ...
    Each split becomes a separate FAISS query for better retrieval.
    """
    if not snippet:
        return []
    parts = [p.strip() for p in snippet.split("...")]
    return [p for p in parts if len(p) > 10]   # ignore very short fragments


def get_entity_queries(target_entity_info: dict) -> list:
    """
    From target entity extraction result, return each entity text as a separate query.
    e.g., entities = [{'text': 'Congressional'}, {'text': 'Obamacare'}, {'text': 'US'}]
    -> ['Congressional', 'Obamacare', 'US']
    Individual entity queries improve retrieval for short named entities.
    """
    return [e["text"] for e in target_entity_info.get("entities", []) if e.get("text")]


def multi_query_retrieve(index, chunks: list, queries: list, top_k: int = 3) -> list:
    """
    Run multiple queries against the same FAISS index and combine results.
    Deduplicates by chunk text; keeps the best score per chunk.
    Returns list sorted by descending score.
    """
    if index is None or not chunks or not queries:
        return []

    best = {}   # chunk_text -> best score
    for q in queries:
        if not q or not q.strip():
            continue
        results = query_index(index, chunks, q, top_k=top_k)
        for r in results:
            txt = r["text"]
            if txt not in best or r["score"] > best[txt]:
                best[txt] = r["score"]

    return sorted(
        [{"text": txt, "score": score} for txt, score in best.items()],
        key=lambda x: -x["score"]
    )


def apply_threshold_logic(retrieved_chunks: list, original_snippet: str) -> dict:
    """
    Apply similarity threshold logic from meeting:
      > 0.7  -> swap: use chunk as primary evidence (snippet replaced)
      0.5-0.7 -> keep both chunk and original snippet
      < 0.5  -> fall back to original snippet only (chunk ignored)

    Snippet is ALWAYS kept as baseline/fallback - never removed entirely.
    The snippet should always be there as a safety net.

    Returns dict with:
      best_chunk, best_score, primary_evidence, evidence_source,
      all_chunks_above_mid (for entity extraction)
    """
    if not retrieved_chunks:
        return {
            "best_chunk":          "",
            "best_score":          0.0,
            "primary_evidence":    original_snippet,
            "evidence_source":     "snippet_only",
            "all_chunks_above_mid": [],
        }

    best  = retrieved_chunks[0]   # sorted descending
    score = best["score"]
    chunk = best["text"]
    above_mid = [c for c in retrieved_chunks if c["score"] > SCORE_MID]

    if score > SCORE_HIGH:
        # Very good match -> swap snippet with chunk as evidence
        return {
            "best_chunk":          chunk,
            "best_score":          score,
            "primary_evidence":    chunk,
            "evidence_source":     "chunk_only",
            "all_chunks_above_mid": above_mid,
        }
    elif score > SCORE_MID:
        # Good match -> keep both; original snippet as fallback
        combined = chunk + "\n\n[Original snippet fallback]:\n" + original_snippet
        return {
            "best_chunk":          chunk,
            "best_score":          score,
            "primary_evidence":    combined,
            "evidence_source":     "chunk_and_snippet",
            "all_chunks_above_mid": above_mid,
        }
    else:
        # Poor match -> ignore chunk, use original snippet only
        return {
            "best_chunk":          "",
            "best_score":          score,
            "primary_evidence":    original_snippet,
            "evidence_source":     "snippet_only",
            "all_chunks_above_mid": [],
        }


def extract_entities_from_chunks(chunks_above_mid: list, target_type: str) -> list:
    """
    From all retrieved chunks above mid threshold, extract entities of the target type.
    Returns deduplicated list of entity dicts (text, label, score).
    """
    labels = ENTITY_TYPE_TO_GLINER_LABELS.get(target_type, ENTITY_TYPE_TO_GLINER_LABELS["UNKNOWN"])
    all_entities = []
    seen = set()
    for chunk in chunks_above_mid:
        spans = _gliner_predict(chunk["text"], labels, GLINER_THRESHOLD)
        for span in spans:
            norm = span["text"].strip().lower()
            if norm and norm not in seen and norm not in _STOPWORDS and len(norm) > 1:
                seen.add(norm)
                all_entities.append({
                    "text":  span["text"].strip(),
                    "label": span["label"],
                    "score": round(float(span["score"]), 3),
                })
    return all_entities


def compare_entity_sets(target_entities: list, chunk_entities: list) -> dict:
    """
    Compare target entities (from QT question) with entities found in retrieved chunks.
    Uses subset / superset / intersection / union comparisons.

    Logic:
      - Chunk entities match target -> more likely TRUE
      - Good chunk score + missing target entities -> CONFLICTING / possibly FALSE
      - No good chunk + missing entities -> CONFLICTING

    Note: High similarity does NOT mean the chunk supports the claim.
    A contradicting chunk with high similarity is VALUABLE evidence (can confirm FALSE).
    Example: Yellowstone question -> chunk says 'NOT Yellowstone national park' -> confirms FALSE.

    Returns:
      verdict: TRUE_SIGNAL | CONFLICTING | INSUFFICIENT
    """
    target_norms = {e.lower().strip() for e in target_entities if e}
    chunk_norms  = {e["text"].lower().strip() for e in chunk_entities if e.get("text")}

    if not target_norms:
        return {"verdict": "INSUFFICIENT", "reason": "no_target_entities",
                "match_ratio": 0.0, "intersection": [], "missing": [], "extra": []}

    intersection = target_norms & chunk_norms
    missing      = target_norms - chunk_norms
    extra        = chunk_norms  - target_norms
    match_ratio  = len(intersection) / len(target_norms)

    if match_ratio >= 0.8:
        verdict, reason = "TRUE_SIGNAL",   "entities_match"
    elif intersection and missing:
        verdict, reason = "CONFLICTING",   "partial_match_entities_missing"
    elif not intersection:
        verdict, reason = "CONFLICTING",   "no_matching_entities"
    else:
        verdict, reason = "INSUFFICIENT",  "low_match_ratio"

    return {
        "verdict":     verdict,
        "reason":      reason,
        "match_ratio": round(match_ratio, 3),
        "intersection": list(intersection),
        "missing":      list(missing),
        "extra":        list(extra),
    }


print("Multi-query retrieval helpers defined.")


Multi-query retrieval helpers defined.


## 8. Checkpoint Save/Load Helpers

**Why:** Colab sessions expire. Saving progress every N records means you never re-process
what's already done. Just re-run the pipeline cell and it will pick up from where it left off.


In [ ]:
def save_checkpoint(records: list, path: str = CHECKPOINT_PATH):
    with open(path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  [Checkpoint] Saved {len(records)} records -> {path}")

def load_checkpoint(path: str = CHECKPOINT_PATH) -> list:
    if not os.path.exists(path):
        print(f"  [Checkpoint] No checkpoint at {path}. Starting fresh.")
        return []
    with open(path) as f:
        records = json.load(f)
    print(f"  [Checkpoint] Loaded {len(records)} previously processed records from {path}")
    return records

def get_processed_keys(records: list) -> set:
    return {(r["claim_id"], r["q_idx"]) for r in records}

print("Checkpoint helpers defined.")


Checkpoint helpers defined.


## 9. Updated `process_one` -- Multi-Query + Threshold + Entity Comparison

Changes from previous version:
- **Multi-query**: uses snippet splits + individual entities + question as FAISS queries
- **Threshold logic**: evidence_source reflects swap/keep-both/fallback decision
- **Entity extraction from chunks**: extracts entities from all chunks with score > 0.5
- **Entity set comparison**: compares against target entities for TRUE/CONFLICTING signal
- **Stores all retrieved chunks above mid threshold** for downstream analysis


In [ ]:
def process_one(args):
    sid, q_idx, item, match = args
    our_q    = item["question"]
    sr_list  = item["search_results"]
    qt_q     = match["matched_qt_question"]
    m_score  = match["score"]
    top_link = sr_list[0].get("link", "") if sr_list else ""

    # Original snippet (Google provides text where query matches, with ... separators)
    original_snippet = sr_list[0].get("snippet", "") if sr_list else ""

    # Fetch article
    article_text, used_url, fetch_source = fetch_best_article(our_q, top_link, sr_list)

    # Entity extraction (question-level)
    our_ents     = scan_all_entities(our_q)
    qt_ents      = scan_all_entities(qt_q) if qt_q else []
    missing_ents = compute_missing_entities(qt_ents, our_ents)
    target_info  = extract_target_entity(our_q, qt_q or "")
    target_entity_texts = [e["text"] for e in target_info.get("entities", []) if e.get("text")]

    if article_text:
        chunks = chunk_text(article_text)
        index, chunks = build_faiss_index(chunks)

        # --- NEW: Build multi-query list ---
        # 1. Snippet splits (split original snippet by ...)
        snippet_queries = split_snippet_queries(original_snippet)
        # 2. Individual target entities as separate queries
        entity_queries  = get_entity_queries(target_info)
        # 3. Original question (for coverage)
        all_queries = [our_q] + snippet_queries + entity_queries
        # Deduplicate while preserving order
        seen_q, deduped_queries = set(), []
        for q in all_queries:
            norm_q = q.lower().strip()
            if norm_q not in seen_q:
                seen_q.add(norm_q)
                deduped_queries.append(q)

        # --- NEW: Multi-query retrieval ---
        retrieved_chunks = multi_query_retrieve(index, chunks, deduped_queries, top_k=3)

        # --- NEW: Threshold logic ---
        threshold_result = apply_threshold_logic(retrieved_chunks, original_snippet)

        # --- NEW: Entity extraction from good chunks (score > 0.5) ---
        chunk_entities = extract_entities_from_chunks(
            threshold_result["all_chunks_above_mid"],
            target_info["entity_type"]
        )

        # --- NEW: Entity set comparison ---
        entity_comparison = compare_entity_sets(target_entity_texts, chunk_entities)

    else:
        snippet_queries   = split_snippet_queries(original_snippet)
        entity_queries    = get_entity_queries(target_info) if 'target_info' in dir() else []
        deduped_queries   = []
        retrieved_chunks  = []
        threshold_result  = {
            "best_chunk": "", "best_score": 0.0,
            "primary_evidence": original_snippet,
            "evidence_source": "snippet_only",
            "all_chunks_above_mid": []
        }
        chunk_entities    = []
        entity_comparison = {"verdict": "INSUFFICIENT", "reason": "no_article",
                             "match_ratio": 0.0, "intersection": [],
                             "missing": list({e.lower() for e in target_entity_texts}),
                             "extra": []}

    # Legacy entity presence flags (kept for backward compatibility)
    top_chunk = threshold_result["best_chunk"]
    our_present,     _ = any_entity_in_text([e[0] for e in our_ents],     top_chunk)
    qt_present,      _ = any_entity_in_text([e[0] for e in qt_ents],      top_chunk)
    missing_present, _ = any_entity_in_text([e[0] for e in missing_ents], top_chunk)

    # High-level flag
    ev_verdict = entity_comparison["verdict"]
    best_score = threshold_result["best_score"]

    if not qt_q:
        flag = "NO_QT_MATCH"
    elif best_score > SCORE_HIGH and ev_verdict == "TRUE_SIGNAL":
        flag = "LIKELY_TRUE"
    elif best_score > SCORE_HIGH and ev_verdict == "CONFLICTING":
        flag = "POSSIBLY_FALSE"   # good chunk but entities missing/contradicting
    elif best_score > SCORE_MID and ev_verdict == "CONFLICTING":
        flag = "CONFLICTING"
    elif qt_present or our_present:
        flag = "TARGET_ENTITY_PRESENT"
    else:
        flag = "TARGET_ENTITY_MISSING"

    return {
        # Identification
        "claim_id":         sid,
        "claim":            claims[sid],
        "label":            labels[sid],
        "category":         categories[sid],
        "taxonomy":         taxonomy[sid],
        "q_idx":            q_idx,
        # Question matching
        "our_question":     our_q,
        "matched_qt_q":     qt_q or "",
        "match_score":      m_score,
        # Fetch info
        "used_url":         used_url or "",
        "fetch_source":     fetch_source,
        "original_snippet": original_snippet,
        # NEW: Multi-query info
        "snippet_queries":    snippet_queries,
        "entity_queries":     entity_queries,
        "num_queries_used":   len(deduped_queries),
        # NEW: Retrieval + threshold results
        "best_chunk":           threshold_result["best_chunk"],
        "best_score":           threshold_result["best_score"],
        "primary_evidence":     threshold_result["primary_evidence"],
        "evidence_source":      threshold_result["evidence_source"],
        "top_chunks_above_mid": threshold_result["all_chunks_above_mid"],
        # NEW: Entity analysis
        "target_entity":       target_info["best_entity"],
        "entity_type":         target_info["entity_type"],
        "extraction_status":   target_info["status"],
        "our_entities":        our_ents,
        "qt_entities":         qt_ents,
        "missing_entities":    missing_ents,
        "chunk_entities":      chunk_entities,
        "entity_comparison":   entity_comparison,
        # Legacy flags
        "our_present":         our_present,
        "qt_present":          qt_present,
        "missing_recovered":   missing_present,
        "flag":                flag,
    }

print("process_one (updated) defined.")


process_one (updated) defined.


## 10. Full Pipeline with Checkpoint Saving

- Loads existing checkpoint on startup (if any).
- Skips already-processed (claim_id, q_idx) pairs.
- Saves checkpoint every `SAVE_EVERY` newly completed records.
- Safe to re-run after interruption -- no re-processing of completed work.


In [ ]:
SAVE_EVERY  = 50   # checkpoint interval
MAX_WORKERS = 20   # threads for HTTP fetching

# Build full args list
all_args = [
    (sid, q_idx, item, match)
    for sid in sample_ids
    for q_idx, (item, match) in enumerate(zip(our_evidences[sid], match_results[sid]))
]

# Load checkpoint
output_records = load_checkpoint(CHECKPOINT_PATH)
processed_keys = get_processed_keys(output_records)

# Filter already-processed
remaining_args = [a for a in all_args if (a[0], a[1]) not in processed_keys]
print(f"Total: {len(all_args)}  |  Done: {len(processed_keys)}  |  Remaining: {len(remaining_args)}")

# Process
new_records = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one, args): args for args in remaining_args}
    with tqdm(total=len(remaining_args), desc="Processing", unit="q") as pbar:
        for future in concurrent.futures.as_completed(futures):
            try:
                rec = future.result()
                new_records.append(rec)
                output_records.append(rec)
                if len(new_records) % SAVE_EVERY == 0:
                    save_checkpoint(output_records, CHECKPOINT_PATH)
            except Exception as e:
                args = futures[future]
                print(f"  Error claim={args[0]} q={args[1]}: {e}")
            pbar.update(1)

# Final save
save_checkpoint(output_records, CHECKPOINT_PATH)
output_records.sort(key=lambda r: (int(r["claim_id"]), r["q_idx"]))
print(f"Done. {len(output_records)} total records.")


## 11. Empirical Analysis: Snippet vs Retrieved Chunk



In [ ]:
import pandas as pd

output_records = load_checkpoint('/content/checkpoint_records (1).json')
df_analysis = pd.DataFrame(output_records)

HIGH_SIM_THRESHOLD = 0.6   # change to 0.7 to focus on clearest cases

high_sim_df = df_analysis[df_analysis["best_score"] >= HIGH_SIM_THRESHOLD].copy()
print(f"Cases with chunk score >= {HIGH_SIM_THRESHOLD}: {len(high_sim_df)}")

sample_n = min(20, len(high_sim_df))
sample_df = high_sim_df.sample(n=sample_n, random_state=42) if len(high_sim_df) >= sample_n else high_sim_df
print(f"Sampled {len(sample_df)} cases for manual review")

for i, (_, row) in enumerate(sample_df.iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"Case {i}/{len(sample_df)}")
    print(f"Claim ID: {row['claim_id']}  |  Label: {row['label']}  |  Chunk Score: {row['best_score']:.3f}")
    print(f"Claim:    {row['claim']}")
    print(f"Question: {row['our_question']}")
    print(f"Evidence Source: {row['evidence_source']}")

    print(f"\n--- Original Snippet ---")
    print(row.get('original_snippet') or '[empty]')

    print(f"\n--- Retrieved Chunk (score={row['best_score']:.3f}) ---")
    print(row.get('best_chunk') or '[no chunk retrieved - below threshold]')

    ec = row.get('entity_comparison') or {}
    if isinstance(ec, dict):
        print(f"\n--- Entity Comparison ---")
        print(f"  Verdict:    {ec.get('verdict','')}")
        print(f"  Reason:     {ec.get('reason','')}")
        print(f"  Matching:   {ec.get('intersection','')}")
        print(f"  Missing:    {ec.get('missing','')}")

    print(f"\n--- Flag: {row['flag']} ---")
    print(f"  Snippet queries used: {row.get('snippet_queries', [])}")
    print(f"  Entity queries used:  {row.get('entity_queries', [])}")
    print(f"  Total queries:        {row.get('num_queries_used', 0)}")


  [Checkpoint] Loaded 46245 previously processed records from /content/checkpoint_records (1).json
Cases with chunk score >= 0.6: 28941
Sampled 20 cases for manual review

Case 1/20
Claim ID: 7189  |  Label: False  |  Chunk Score: 0.887
Claim:    Deaths counted as Covid-19 related within 20 days of contracting the virus, no matter what other factors were involved?
Question: Which disease was involved in more than half of the deaths?
Evidence Source: chunk_only

--- Original Snippet ---
Jul 5, 2024 ... Centers for Disease ... More than half of firearm-related deaths were suicides and more than four out of every 10 were firearm homicides.

--- Retrieved Chunk (score=0.887) ---
More than half of firearm-related deaths were suicides and more than four out of every 10 were firearm homicides.1 More people suffer nonfatal firearm-related injuries than die. More than seven out of every 10 medically treated firearm injuries are from firearm-related assaults. Nearly two out of every 10 are from 

## 12. Export Enriched JSON

In [ ]:
from collections import defaultdict

export = defaultdict(lambda: {
    "claim": "", "label": "", "category": "", "taxonomy": "", "questions": []
})

for rec in output_records:
    sid = rec["claim_id"]
    export[sid]["claim"]    = rec["claim"]
    export[sid]["label"]    = rec["label"]
    export[sid]["category"] = rec["category"]
    export[sid]["taxonomy"] = rec["taxonomy"]
    export[sid]["questions"].append({
        "q_idx":               rec["q_idx"],
        "our_question":        rec["our_question"],
        "matched_qt_q":        rec["matched_qt_q"],
        "match_score":         rec["match_score"],
        "used_url":            rec["used_url"],
        "fetch_source":        rec["fetch_source"],
        # Evidence
        "original_snippet":    rec["original_snippet"],
        "snippet_queries":     rec["snippet_queries"],
        "entity_queries":      rec["entity_queries"],
        "num_queries_used":    rec["num_queries_used"],
        "best_chunk":          rec["best_chunk"],
        "best_score":          rec["best_score"],
        "primary_evidence":    rec["primary_evidence"],
        "evidence_source":     rec["evidence_source"],
        "top_chunks_above_mid": rec["top_chunks_above_mid"],
        # Entities
        "target_entity":       rec["target_entity"],
        "entity_type":         rec["entity_type"],
        "our_entities":        rec["our_entities"],
        "qt_entities":         rec["qt_entities"],
        "missing_entities":    rec["missing_entities"],
        "chunk_entities":      rec["chunk_entities"],
        "entity_comparison":   rec["entity_comparison"],
        # Flags
        "our_present":         rec["our_present"],
        "qt_present":          rec["qt_present"],
        "missing_recovered":   rec["missing_recovered"],
        "flag":                rec["flag"],
    })

with open(FINAL_OUTPUT_PATH, "w") as f:
    json.dump(dict(export), f, indent=2)
print(f"Saved {len(export)} claim records to '{FINAL_OUTPUT_PATH}'")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)
files.download(CHECKPOINT_PATH)


Saved 15415 claim records to '/content/claim_verification_results_v2.json'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Summary Statistics

In [ ]:
import pandas as pd

df = pd.DataFrame(output_records)

print("=== Flag Distribution ===")
print(df["flag"].value_counts().to_string())

print("\n=== Evidence Source Distribution ===")
print(df["evidence_source"].value_counts().to_string())

print("\n=== Fetch Source Distribution ===")
print(df["fetch_source"].value_counts().to_string())

print("\n=== Chunk Score Distribution ===")
bins   = [0, 0.3, 0.5, 0.7, 1.0]
labels_b = ["<0.3", "0.3-0.5", "0.5-0.7", ">0.7"]
df["score_bin"] = pd.cut(df["best_score"], bins=bins, labels=labels_b)
print(df["score_bin"].value_counts().sort_index().to_string())

print("\n=== Entity Comparison Verdict Distribution ===")
verdicts = df["entity_comparison"].apply(
    lambda x: x.get("verdict", "N/A") if isinstance(x, dict) else "N/A"
)
print(verdicts.value_counts().to_string())

print("\n=== Missing Entity Recovery Rate ===")
has_missing = df[df["missing_entities"].map(len) > 0]
recovered   = has_missing["missing_recovered"].sum()
print(f"  {recovered}/{len(has_missing)} questions recovered at least one missing entity ({100*recovered/max(len(has_missing),1):.1f}%)")

print("\n=== Flag by Taxonomy ===")
print(df.groupby("taxonomy")["flag"].value_counts().unstack(fill_value=0).to_string())

print("\n=== Avg Queries Used per Fetched Article ===")
fetched = df[df["num_queries_used"] > 0]
print(f"  Mean: {fetched['num_queries_used'].mean():.1f}  Max: {fetched['num_queries_used'].max()}")


## Generate Signals

In [ ]:
import json
from collections import defaultdict

# ── Load checkpoint ───────────────────────────────────────────
with open("/content/drive/MyDrive/claim_verification_analysis_updated_v2/checkpoint_records_merged.json") as f:
    records = json.load(f)

# ── Build lookup: claim_id -> list of question records ────────
claim_groups = defaultdict(list)
for r in records:
    claim_groups[r["claim_id"]].append(r)

# ── Compute entity signal per claim ───────────────────────────
def compute_entity_signal(recs):
    """
    Collect all target entities across all questions for this claim.
    Collect all chunk_entities extracted from retrieved chunks.
    Do one set comparison across the whole claim.
    """
    # All target entities from QT questions (what we're looking for)
    target_set = set()
    for r in recs:
        for e in r.get("qt_entities", []):
            text = e[0] if isinstance(e, list) else e.get("text", "")
            if text:
                target_set.add(text.lower().strip())

    # All entities actually found in retrieved chunks
    chunk_set = set()
    for r in recs:
        for e in r.get("chunk_entities", []):
            text = e.get("text", "") if isinstance(e, dict) else str(e)
            if text:
                chunk_set.add(text.lower().strip())

    if not target_set:
        return "some_present"   # no target to compare against

    intersection = target_set & chunk_set
    match_ratio  = len(intersection) / len(target_set)

    if match_ratio == 1.0:   return "all_present"
    elif match_ratio == 0.0: return "none_present"
    else:                    return "some_present"
entity_signal_map = {
    cid: compute_entity_signal(recs)
    for cid, recs in claim_groups.items()
}

# ── Build chunk lookup: claim_id -> sorted list of question dicts ─
chunk_lookup = {}
for cid, recs in claim_groups.items():
    sorted_recs = sorted(recs, key=lambda r: r["q_idx"])
    chunk_lookup[cid] = [
        {
            "question": r["our_question"],
            "top_k_doc": (
                [c["text"] for c in r["top_chunks_above_mid"]]
                if r.get("top_chunks_above_mid")
                else [r["original_snippet"]]   # fallback
            )
        }
        for r in sorted_recs
    ]

# ── Load and enrich converted.json ───────────────────────────
INPUT_PATH  = "/content/drive/MyDrive/claim_verification_analysis_updated_v2/converted.json"
OUTPUT_PATH = "/content/drive/MyDrive/claim_verification_analysis_updated_v2/converted_with_signal.json"

with open(INPUT_PATH) as f:
    data = json.load(f)

# ── Alignment check ───────────────────────────────────────────
checkpoint_ids = set(entity_signal_map.keys())
data_ids       = set(str(i) for i in range(len(data)))
print(f"Checkpoint claims : {len(checkpoint_ids)}")
print(f"converted.json    : {len(data)}")

missed = 0
for i, item in enumerate(data):
    cid = str(i)   # claim_id in checkpoint is row index as string
    if cid in chunk_lookup:
        item["reranked_our_evidences"] = chunk_lookup[cid]
        item["entity_signal"]          = entity_signal_map[cid]
    else:
        # keep original evidences, set signal to unknown
        item["entity_signal"] = "some_present"
        missed += 1

print(f"Claims updated    : {len(data) - missed}")
print(f"Fallback (missed) : {missed}")

with open(OUTPUT_PATH, "w") as f:
    json.dump(data, f, indent=2)
print(f"\nSaved → {OUTPUT_PATH}")

# ── Sanity check one record ───────────────────────────────────
print("\n=== Sample record ===")
print(json.dumps(data[0], indent=2)[:800])

Checkpoint claims : 15478
converted.json    : 15478
Claims updated    : 15478
Fallback (missed) : 0

Saved → /content/drive/MyDrive/claim_verification_analysis_updated_v2/converted_with_signal.json

=== Sample record ===
{
  "claim": "\"The non-partisan Congressional Budget Office concluded ObamaCare will cost the U.S. more than 800,000 jobs.\"",
  "label": "Conflicting",
  "category": "test",
  "taxonomy": "Statistical",
  "reranked_our_evidences": [
    {
      "question": "Who concluded ObamaCare will cost the U.S. 800,000 jobs?",
      "top_k_doc": [
        "The Congressional Budget Office says Obamacare will result in 800,000 fewer American jobs. Many business owners express concerns and confusion about what how the law will impact their business and their bottom line. \u201cI\u2019ve met with thousands of South Carolina business owners and job creators in the two years since this was passed into law,\u201d said Graham.",
        "Thus far, the Obama Administration has issued at 

## for temporal NEw evidences
Can be ignored. Earlier i did not worked with refined evidences of temporal taxonomy. So i corrected that using following code.

In [ ]:
import pandas as pd
df = pd.read_json("Processed_complete_dataset_with_new_temporal_evidences.json")
df.head()

,claim,quantemp_evidences,label,Category,our_evidences,taxonomy,reranked_our_evidences
0,"""The non-partisan Congressional Budget Office ...",[{'questions': 'did the congressional budget o...,Conflicting,test,[{'question': 'Who concluded ObamaCare will co...,Statistical,[{'questions': 'Who concluded ObamaCare will c...
1,"""More than 50 percent of immigrants from (El S...",[{'questions': 'do more than 50 percent of imm...,True,test,[{'question': 'Which country has over 50 perce...,Statistical,[{'questions': 'Which country has over 50 perc...
2,"""[In 2014-2015] coverage for the rotavirus vac...",[{'questions': 'did the coverage for the rotav...,False,test,[{'question': 'Which vaccine had 91.5% coverag...,Statistical,[{'questions': 'Which vaccine had 91.5% covera...
3,Labour took £1.5 million from Just Stop Oil.,[{'questions': 'did labour receive a donation ...,False,test,[{'question': 'Who took more than £1.5 million...,Statistical,[{'questions': 'Who took more than £1.5 millio...
4,McDonald's has announced that everyone who sha...,[{'questions': 'is it true that mcdonald's has...,False,test,[{'question': 'Who shares this link will recei...,Statistical,[{'questions': 'Who shares this link will rece...


In [ ]:
df.shape

(15415, 7)

In [ ]:
df2 = df[df.taxonomy == 'Temporal']
df2

,claim,quantemp_evidences,label,Category,our_evidences,taxonomy,reranked_our_evidences
11285,Kyiv cut off gas in the territory of Luhansk ...,[{'questions': 'did kyiv cut off gas in the te...,False,train,[{'question': 'Which country cut off gas in th...,Temporal,[{'question': 'Which country cut off gas in th...
11286,Photo showing a car carrying russian diplomat...,[{'questions': 'is there a photo showing a car...,False,train,[{'question': 'Which car carrying russian dipl...,Temporal,[{'question': 'Which car carrying russian dipl...
11287,Photograph showing jets at the Forum in Davos...,[{'questions': 'is the forum in davos hosting ...,False,train,[{'question': 'Which photograph shows jets at ...,Temporal,[{'question': 'Which photograph shows jets at ...
11288,There are photos of documents of captured sol...,[{'questions': 'are there photos of documents ...,False,train,[{'question': 'Which photos of captured soldie...,Temporal,[{'question': 'Which photos of captured soldie...
11289,There is a photo of russian Ka-52 helicopter ...,[{'questions': 'is there a photo of a russian ...,False,train,[{'question': 'Which russian Ka-52 helicopter ...,Temporal,[{'question': 'Which russian Ka-52 helicopter ...
...,...,...,...,...,...,...,...
15410,“Tim Ryan when he ran for president two years ...,[{'questions': 'did tim ryan run for president...,Half True/False,train,[{'question': 'Which politician supported bann...,Temporal,[{'question': 'Which politician supported bann...
15411,“Tim Tebow says he is wearing #85 to honor his...,[{'questions': 'is tim tebow wearing #85 to ho...,False,test,[{'question': 'Who says he is wearing #85 to h...,Temporal,[{'question': 'Who says he is wearing #85 to h...
15412,“Wladimir Klitschko has auctioned off his 1996...,[{'questions': 'did wladimir klitschko auction...,Conflicting,test,[{'question': 'Who has auctioned off his Atlan...,Temporal,[{'question': 'Who has auctioned off his Atlan...
15413,“World Health Organization wants to BAN all wo...,[{'questions': 'is the world health organizati...,False,train,[{'question': 'Which organization wants to BAN...,Temporal,[{'question': 'Which organization wants to BAN...


In [ ]:
df2.to_json("temporal_dataset_with_new_evidences.json", indent=2, orient='records')

In [ ]:
import pandas as pd
df4 = pd.read_json("/content/claim_verification_results_v2.json")
df4

,0,1,2,3,4,5,6,7,8,9,...,15465,15468,15469,15470,15471,15472,15473,15475,15476,15477
claim,"""The non-partisan Congressional Budget Office ...","""More than 50 percent of immigrants from (El S...",Photograph showing jets at the Forum in Davos...,"""[In 2014-2015] coverage for the rotavirus vac...",There is a photo of russian Ka-52 helicopter ...,Labour took £1.5 million from Just Stop Oil.,McDonald's has announced that everyone who sha...,meat and dairy products will be banned by 203...,russia destroyed the S-300 air defense system...,russian occupiers try to push out stuck equip...,...,A U.S. tourist in Saudi Arabia was arrested fo...,"""Last month, more Americans stopped looking fo...","""In states where the federal government helps ...","Teeth whitening can be done using carrot, turm...",The 2020 election could not have been fair bec...,Hamburger Piraten-Prozess – Leben die Somalier...,"Mark Zuckerberg is giving $1,000 away to Faceb...","""CNN to permanently close its doors as ratings...","A recent study found ""that cities where Uber o...","The Biden administration ""published a study co..."
label,Conflicting,True,False,False,False,False,False,False,False,False,...,False,Half True/False,Half True/False,False,False,Half True/False,False,False,Half True/False,True
category,test,test,train,test,train,test,test,test,train,train,...,validation,validation,validation,validation,validation,validation,validation,validation,validation,validation
taxonomy,Statistical,Statistical,Temporal,Statistical,Temporal,Statistical,Statistical,Temporal,Temporal,Temporal,...,Statistical,Comparison,Statistical,Interval,Statistical,Interval,Statistical,Statistical,Comparison,Statistical
questions,"[{'q_idx': 0, 'our_question': 'Who concluded O...","[{'q_idx': 0, 'our_question': 'Which country h...","[{'q_idx': 0, 'our_question': 'Which photograp...","[{'q_idx': 0, 'our_question': 'Which vaccine h...","[{'q_idx': 0, 'our_question': 'Which russian K...","[{'q_idx': 0, 'our_question': 'Who took more t...","[{'q_idx': 0, 'our_question': 'Who shares this...","[{'q_idx': 0, 'our_question': 'Which products ...","[{'q_idx': 0, 'our_question': 'Which country d...","[{'q_idx': 0, 'our_question': 'Which country’s...",...,"[{'q_idx': 0, 'our_question': 'Which tourist r...","[{'q_idx': 0, 'our_question': 'Which month had...","[{'q_idx': 0, 'our_question': 'Which state has...","[{'q_idx': 0, 'our_question': 'Which product c...","[{'q_idx': 0, 'our_question': 'Which county ag...","[{'q_idx': 0, 'our_question': 'Which country h...","[{'q_idx': 0, 'our_question': 'Mark Zuckerberg...","[{'q_idx': 0, 'our_question': '""CNN to permane...","[{'q_idx': 0, 'our_question': 'Which city has ...","[{'q_idx': 0, 'our_question': 'Which of the 5 ..."


In [ ]:
df4.shape

(5, 12380)